# PokerTilt — Wins and Bad-Beat Investigations (100NL)

**Standalone analysis notebook for the `Poker-hand-tilt` repository.** This is a **separate file** and does **not edit, run, or replace `PokerTilt.ipynb`**. It reuses the methodology from our earlier wins and bad-beat experiments rather than changing the existing model pipeline.

**What it investigates**
- **Wins:** recent large wins, consecutive wins, and being far ahead in a session; compares rates of unusually large betting.
- **Bad beats:** fully observable heads-up all-in losses where the losing player was strongly favored; compares subsequent anomaly rates with other large all-in losses.

**Important distinctions:** The wins experiment's top-5% score cutoff differs from the bad-beat `s > 2` cutoff. The bad-beat equity cutoff is configurable and defaults to **70%**, following the earlier script; some presentations used **80%**. Run the notebook to calculate findings; no slide percentages are hard-coded.

In [ ]:
# Install only the external dependencies for this standalone notebook.
%pip -q install numpy pandas pyarrow scikit-learn statsmodels matplotlib treys

from pathlib import Path
import os

# If your data is stored in a different folder, change ONLY these input paths.
BETS_100NL = 'bets.parquet'
DECISIONS_100NL = 'decisions.parquet'

assert Path(BETS_100NL).is_file(), f'Upload or mount the original 100NL {BETS_100NL}'
assert Path(DECISIONS_100NL).is_file(), f'Upload or mount the original 100NL {DECISIONS_100NL}'
print('Inputs found:', BETS_100NL, DECISIONS_100NL)


## 1. Investigating wins


In [ ]:
def run_wins_100nl(bets_path="bets.parquet", decisions_path="decisions.parquet", output_dir="wins_100NL_outputs"):
    # Original Wins_Investigation_100NL.ipynb cell 3
    from pathlib import Path
    import sys,time,json
    import numpy as np,pandas as pd
    from sklearn.model_selection import GroupKFold
    from sklearn.ensemble import HistGradientBoostingRegressor
    import statsmodels.formula.api as smf
    BASE=Path(output_dir)
    BASE.mkdir(parents=True, exist_ok=True)
    T=time.time()
    def say(*args): print(f'[{time.time()-T:.1f}s]',*args,flush=True)
    B_COLS=['player','hand_id','street_idx','action_idx','ts','session_key','bet_to','committed_before','stack_before','spr','bet_to_bb','pot_before_bb','stack_before_bb','to_call_bb','n_active','n_players','seat_idx','is_facing_bet','cum_net_bb','loss_streak','hand_in_session']
    D_COLS=['player','hand_id','session_key','ts','hand_net_bb','action','hand_in_session']
    B=pd.read_parquet(bets_path,columns=B_COLS)
    D=pd.read_parquet(decisions_path,columns=D_COLS)
    assert len(B)==518362 and len(D)==2926902
    say('Loaded',len(B),'bets and',len(D),'decisions')

    # Original Wins_Investigation_100NL.ipynb cell 5
    B['is_all_in']=(B.bet_to-B.committed_before)>=B.stack_before-1e-9
    B=B.sort_values(['hand_id','action_idx'],kind='stable')
    B['n_bets_before']=B.groupby(['hand_id','street_idx'],sort=False).cumcount()
    B=B.sort_values('ts',kind='stable').reset_index(drop=True)
    forced=B.is_all_in & (B.spr<1)
    B=B.loc[~forced].reset_index(drop=True)
    say('Removed forced all-ins',int(forced.sum()))
    FEATURES=['street_idx','pot_before_bb','stack_before_bb','to_call_bb','spr','n_active','n_players','seat_idx','is_facing_bet','n_bets_before']
    X=B[FEATURES]; y=np.log(B.bet_to_bb)
    B['pred_log']=np.nan
    for fold,(train,test) in enumerate(GroupKFold(5).split(X,y,groups=B.player),1):
        model=HistGradientBoostingRegressor(random_state=0)
        model.fit(X.iloc[train],y.iloc[train])
        B.loc[test,'pred_log']=model.predict(X.iloc[test]);say('Fit player-held-out fold',fold)
    B['resid']=y-B.pred_log
    # Match the original uploaded notebook's eligibility and calibration rules exactly.
    counts=B.player.value_counts()
    eligible=counts.index[counts>=200]
    P=B.loc[B.player.isin(eligible)].copy()
    P['sess_start']=P.groupby('session_key').ts.transform('min')
    P=P.sort_values(['player','sess_start','ts'],kind='stable')
    P['frac']=P.groupby('player').cumcount()/P.groupby('player').player.transform('size')
    P['is_calib']=P.groupby('session_key')['frac'].transform('min')<0.5
    CAL=P[P.is_calib]
    mu=CAL.groupby('player').resid.median()
    sigma=1.4826*(CAL.resid-CAL.player.map(mu)).abs().groupby(CAL.player).median()
    P['mu_p']=P.player.map(mu);P['sigma_p']=P.player.map(sigma)
    P['s']=(P.resid-P.mu_p)/P.sigma_p
    P['personal_log_excess']=P.resid-P.mu_p
    TEST=P.loc[~P.is_calib].copy()
    assert len(TEST)==124058 and len(eligible)==516
    say('Eligible players',len(eligible),'held-out bet/raise decisions',len(TEST))


    # Original Wins_Investigation_100NL.ipynb cell 7
    # Player-hand grain, including hands with no betting action.
    H=(D.assign(played=D.action!='fold')
        .groupby(['player','hand_id'],as_index=False,sort=False)
        .agg(session_key=('session_key','first'),ts=('ts','first'),
             hand_net_bb=('hand_net_bb','first'),played=('played','any'),
             hand_in_session=('hand_in_session','first')))
    PLAYED=H[H.played]
    typical=PLAYED.groupby('player').hand_net_bb.median()
    spread=1.4826*(PLAYED.hand_net_bb-PLAYED.player.map(typical)).abs().groupby(PLAYED.player).median()
    H['big_loss']=H.hand_net_bb<H.player.map(typical)-2*H.player.map(spread)
    H['big_win']=H.hand_net_bb>H.player.map(typical)+2*H.player.map(spread)
    # 50 BB is a prespecified absolute large-win definition, not the slide's 2*MAD event.
    H['very_big_win']=H.hand_net_bb>=50
    H=H.sort_values(['session_key','ts','hand_in_session'],kind='stable').reset_index(drop=True)

    def recent_event(event,k=5):
        previous=event.groupby(H.session_key,sort=False).shift(1,fill_value=False).astype('int8')
        return (previous.groupby(H.session_key,sort=False).rolling(k,min_periods=1).max()
                .reset_index(level=0,drop=True).sort_index().astype(bool))
    for name in ['big_loss','big_win','very_big_win']:
        H[f'after_{name}_5']=recent_event(H[name],5)
    # Number of CONSECUTIVE previously won player-hands; any break or a non-win resets.
    win=H.hand_net_bb.gt(0)
    reset=(~win).groupby(H.session_key,sort=False).cumsum()
    current_streak=win.groupby([H.session_key,reset],sort=False).cumsum()
    H['win_streak_pre']=current_streak.groupby(H.session_key,sort=False).shift(fill_value=0)
    H['win_streak3']=H.win_streak_pre>=3
    # This reconstructed prior session total exactly matches bets.cum_net_bb.
    H['prior_session_net_bb']=H.groupby('session_key',sort=False).hand_net_bb.cumsum()-H.hand_net_bb
    H['ahead_50bb']=H.prior_session_net_bb>=50
    H['ahead_100bb']=H.prior_session_net_bb>=100
    EVENT_COLS=['after_big_loss_5','after_big_win_5','after_very_big_win_5','win_streak3','ahead_50bb','ahead_100bb','prior_session_net_bb','win_streak_pre']
    TEST=TEST.merge(H[['player','hand_id']+EVENT_COLS],on=['player','hand_id'],how='left',validate='many_to_one')
    assert TEST[EVENT_COLS].notna().all().all()
    assert np.isclose(TEST.cum_net_bb,TEST.prior_session_net_bb,atol=1e-6).all()
    TEST['flag_s2']=(TEST.s>2).astype(int)
    flag5_cutoff=TEST.s.quantile(0.95)
    TEST['flag_top5']=(TEST.s>=flag5_cutoff).astype(int)
    TEST['after_big_loss_5']=TEST.after_big_loss_5.astype(int)
    TEST['after_big_win_5']=TEST.after_big_win_5.astype(int)
    TEST['after_very_big_win_5']=TEST.after_very_big_win_5.astype(int)
    TEST['win_streak3']=TEST.win_streak3.astype(int)
    TEST['ahead_50bb']=TEST.ahead_50bb.astype(int)
    TEST['ahead_100bb']=TEST.ahead_100bb.astype(int)
    TEST['recent_big_win_only']=((TEST.after_big_win_5==1)&(TEST.after_very_big_win_5==0)).astype(int)
    TEST['log_hand_index']=np.log1p(TEST.hand_in_session.astype(float))
    TEST['session_group']=pd.factorize(TEST.session_key)[0]

    say('Event counts',H[['big_loss','big_win','very_big_win']].sum().to_dict(), 'top5 cutoff',round(float(flag5_cutoff),3))


    # Original Wins_Investigation_100NL.ipynb cell 9
    # Descriptive rates: each exposure compared to its complement, not mutually exclusive.
    rows=[]
    for col,label in [('after_big_win_5','Large win within 5 hands (player >2 robust SD)'),
                      ('after_very_big_win_5','Very large win within 5 hands (>=50 BB)'),
                      ('win_streak3','At least 3 consecutive winning hands'),
                      ('ahead_50bb','At least +50 BB ahead in session'),
                      ('ahead_100bb','At least +100 BB ahead in session'),
                      ('after_big_loss_5','Large loss within 5 hands (reference)')]:
      for val in (0,1):
        z=TEST.loc[TEST[col]==val]
        rows.append({'condition':label,'indicator':col,'exposed':bool(val),'n_bet_decisions':len(z),
                     'n_players':z.player.nunique(),'n_sessions':z.session_key.nunique(),
                     'mean_bet_bb':z.bet_to_bb.mean(),
                     'flag_rate_s_gt_2':z.flag_s2.mean(), 'flag_rate_matched_top5':z.flag_top5.mean(),
                     'mean_personal_log_excess':z.personal_log_excess.mean(),
                     'geometric_bet_ratio_to_personal_expected':float(np.exp(z.personal_log_excess.mean()))})
    summary=pd.DataFrame(rows)
    summary.to_csv(BASE/'wins_100NL_descriptive.csv',index=False)
    say('Saved descriptive comparison')

    print(summary[["condition","exposed","n_bet_decisions","flag_rate_matched_top5"]].to_string(index=False))

    # Original Wins_Investigation_100NL.ipynb cell 11
    # Adjusted models: all exposures are defined from the past, never the current hand.
    # Large-win-only excludes >=50BB wins to distinguish types of recent win.
    FORM='recent_big_win_only + after_very_big_win_5 + win_streak3 + ahead_100bb + after_big_loss_5 + log_hand_index'
    regressions={}
    for key,formula in [
        ('personal_top5','flag_top5 ~ '+FORM),
        ('personal_s_gt_2','flag_s2 ~ '+FORM),
        ('log_bet_excess','personal_log_excess ~ '+FORM)]:
      say('Fitting',key)
      if key=='log_bet_excess':
        fit=smf.ols(formula,data=TEST).fit(cov_type='cluster',cov_kwds={'groups':TEST.session_group})
      else:
        fit=smf.logit(formula,data=TEST).fit(disp=0,cov_type='cluster',cov_kwds={'groups':TEST.session_group},maxiter=100)
      c=fit.conf_int()
      estimates={}
      for name in fit.params.index:
        if name=='Intercept':continue
        coef=float(fit.params[name]);lo=float(c.loc[name,0]);hi=float(c.loc[name,1])
        estimates[name]={'coefficient':coef,'p':float(fit.pvalues[name]),'ci_coef':[lo,hi],
               'interpretation':('odds ratio' if key!='log_bet_excess' else 'bet ratio'),
               'ratio':float(np.exp(coef)),'ci_ratio':[float(np.exp(lo)),float(np.exp(hi))]}
      regressions[key]={'n':int(fit.nobs),'converged':bool(fit.mle_retvals['converged']) if hasattr(fit,'mle_retvals') else True,'estimates':estimates}
      say(key,'key results',[(n,round(v['ratio'],3),round(v['p'],4)) for n,v in estimates.items()])


    # Original Wins_Investigation_100NL.ipynb cell 13
    # Check alternative event tests, separate win-streak2 and ahead50 absolute levels.
    # Stratified cross-check: among bets NOT also within 5 hands of big loss.
    neutral_results={}
    for var in ['after_big_win_5','after_very_big_win_5','win_streak3','ahead_100bb']:
     for mask,label in [(TEST.after_big_loss_5==0,'excluding_recent_big_loss'),
                        (np.ones(len(TEST),dtype=bool),'all_test_bets')]:
      a=TEST.loc[mask]
      exposed=a.loc[a[var]==1];other=a.loc[a[var]==0]
      neutral_results[var+'_'+label]={'n_exposed':len(exposed),'n_other':len(other),'flag_top5_exposed':float(exposed.flag_top5.mean()),'flag_top5_other':float(other.flag_top5.mean())}

    meta={'source_files':[str(bets_path),str(decisions_path),'100NL win-analysis extension'],
          'rows':{'decisions':int(len(D)),'bets_before_forced_filter':int(len(B)+forced.sum()),'eligible_players':int(len(eligible)),'held_out_bets':int(len(TEST))},
          'all_in_filter_n':int(forced.sum()),'cutoff_s_for_top5':float(flag5_cutoff),
          'counts_in_all_player_hands':{'big_win_2robustSD':int(H.big_win.sum()),'very_big_win_50BB':int(H.very_big_win.sum()),'big_loss_2robustSD':int(H.big_loss.sum())},
          'definitions':{'win_streak3':'at least 3 immediately preceding player-hands with positive net BB',
            'ahead_100bb':'cumulative session net BB before current hand at least 100',
            'very_big_win':'hand_net_bb at least +50 BB; the following five player-hands are exposed',
            'large_win':'player-specific hand_net_bb > median + 2 * 1.4826 * MAD; following five player-hands exposed',
            'flag_top5':'score >= 95th percentile on held-out personal test bets',
            'flag_s_gt_2':'score > 2 with calibration in uploaded notebook; global flag rate differs from slides',
            'clusters':'session_key for robust SE; repeated player sessions not separately controlled'},
          'regressions':regressions,'neutral_subsets':neutral_results}
    with open(BASE/'wins_100NL_results.json','w') as f:json.dump(meta,f,indent=2)
    say('Wrote study outputs')
    # Robustness: participants, rather than sessions, are the independent clusters.
    player_group=pd.factorize(TEST.player)[0]
    robust={}
    for key,formula in [('top5_player_cluster','flag_top5 ~ '+FORM),
                        ('s2_player_cluster','flag_s2 ~ '+FORM),
                        ('excess_player_cluster','personal_log_excess ~ '+FORM)]:
        if key.startswith('excess'):
            fit=smf.ols(formula,data=TEST).fit(cov_type='cluster',cov_kwds={'groups':player_group})
        else:
            fit=smf.logit(formula,data=TEST).fit(disp=0,cov_type='cluster',cov_kwds={'groups':player_group},maxiter=100)
        ci=fit.conf_int()
        robust[key]={v:{'ratio':float(np.exp(fit.params[v])),'p':float(fit.pvalues[v]),
                         'ci_ratio':[float(np.exp(ci.loc[v,0])),float(np.exp(ci.loc[v,1]))]}
                       for v in ['recent_big_win_only','after_very_big_win_5','win_streak3','ahead_100bb','after_big_loss_5']}
        say('Player clustered',key,[(v,round(y['ratio'],3),round(y['p'],5)) for v,y in robust[key].items()])
    meta['robustness_player_clustered']=robust
    with open(BASE/'wins_100NL_results.json','w') as f:json.dump(meta,f,indent=2)

    print("Results written to wins_100NL_results.json and wins_100NL_descriptive.csv")

    # Export the same 100NL personalized scores for bad-beat post-event analysis.
    # This is explicitly the s > 2 flag, not the global top-5% win analysis.
    score_export = TEST[['player','hand_id','session_key','hand_in_session','s','flag_s2']].copy()
    score_export = score_export.rename(columns={'flag_s2':'personal_flag'})
    score_export.to_csv(BASE / 'personal_bet_scores_100NL.csv.gz',index=False,compression='gzip')
    say('Exported 100NL scores for the separate bad-beat experiment')
    return {'win_summary':str(BASE/'wins_100NL_descriptive.csv'),
            'win_results':str(BASE/'wins_100NL_results.json'),
            'badbeat_scores':str(BASE/'personal_bet_scores_100NL.csv.gz'),
            'held_out_bets':len(TEST)}


In [ ]:
# Run the existing 100NL wins experiment on the original Parquet files.
# Change only these two standalone analysis paths if you stored the files elsewhere.
WIN_BETS_100NL = BETS_100NL
WIN_DECISIONS_100NL = DECISIONS_100NL

# On Colab these can be e.g. '/content/drive/MyDrive/capstone/bets.parquet'.
WIN_ADDITION_OUTPUTS = run_wins_100nl(
    bets_path=WIN_BETS_100NL,
    decisions_path=WIN_DECISIONS_100NL,
    output_dir='wins_100NL_outputs',
)
print(WIN_ADDITION_OUTPUTS)


### Wins: view measured rates

The table and figure below are created from the results of the code you actually run. **They are not prefilled with presentation numbers.** The main wins graph uses the **top 5% of personal anomaly scores**, consistent with the earlier wins investigation.

In [ ]:
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt

wins_result = pd.read_csv(WIN_ADDITION_OUTPUTS['win_summary'])
display(wins_result)
print('Wins results metadata:', WIN_ADDITION_OUTPUTS['win_results'])
print('Bad-beat-compatible scored bets:', WIN_ADDITION_OUTPUTS['badbeat_scores'])


## 2. Bad-beat analysis

**Definitions:** Complete revealed hole-card information is required to estimate equity in heads-up all-ins. The losing player must have lost at least **30 BB**. The post-event window is the next **five player-hands**. The **bad-beat equity threshold is configurable**, and the flagging rule uses the personalized **`s > 2`** score. Events without sufficient revealed cards cannot be classified reliably.

In [ ]:
%%writefile badbeat_100NL_analysis.py
"""Strict observational bad-beat analysis for Absolute Poker 100NL (HandHQ PHHS).

This script is designed for the original capstone's 100NL hand histories and
existing decisions.parquet and bets.parquet. It does NOT change preprocessing.

Bad beat (operational): a player losing at least MIN_LOSS_BB in a fully
revealed heads-up all-in hand, having >= BAD_BEAT_EQUITY expected share of
showdown wins at the moment the all-in was effectively called, before river.

Comparison: other fully-revealed heads-up all-in losses of >= MIN_LOSS_BB
where the loser had lower than BAD_BEAT_EQUITY equity at commitment. The
comparison is observational and selected by availability of revealed cards.

Install dependencies: pip install pandas numpy pyarrow treys
"""
from __future__ import annotations

import argparse
import itertools
import json
import random
import re
import tomllib
from pathlib import Path

MIN_LOSS_BB = 30.0
BAD_BEAT_EQUITY = 0.70
WINDOW_HANDS = 5
PREFLOP_TRIALS = 3000

CARD_RE = re.compile(r"^(?:[2-9TJQKA][cdhs]){2}$", re.I)
PL_RE = re.compile(r"^p(\d+)$")
DECK = [r + s for r in '23456789TJQKA' for s in 'cdhs']


def split_cards(s: str) -> list[str]:
    return [s[i:i+2] for i in range(0, len(s), 2)]


def get_showdown_players(hand: dict) -> dict[int, list[str]]:
    """Zero-based seat index -> two visible hole cards (NOT hidden ????)."""
    revealed = {}
    for action in hand.get('actions', []):
        fields = action.split()
        if len(fields) < 3 or fields[1] != 'sm':
            continue
        m = PL_RE.fullmatch(fields[0])
        if m and CARD_RE.fullmatch(fields[2]):
            revealed[int(m.group(1)) - 1] = split_cards(fields[2])
    return revealed


def find_heads_up_allin_lock(hand: dict) -> dict | None:
    """Conservative moment where exactly two live players have locked a bet.

    Detects an all-in followed by a call, or a player calling all-in.
    Skips multiway all-ins and hands where all-in cannot be established.
    Committed amounts are reconstructed like the original PHHS parser.
    """
    stacks0 = [float(x) for x in hand.get('starting_stacks', [])]
    n = len(stacks0)
    if n < 2:
        return None
    blind = [float(v or 0) for v in hand.get('blinds_or_straddles', [])]
    ante = [float(v or 0) for v in hand.get('antes', [])]
    committed = [(blind[i] if i < len(blind) else 0) +
                 (ante[i] if i < len(ante) else 0) for i in range(n)]
    street_commit = [blind[i] if i < len(blind) else 0 for i in range(n)]
    folded = [False] * n
    board = []
    previously_allin = [stacks0[i] - committed[i] <= 1e-5 for i in range(n)]

    for action in hand.get('actions', []):
        parts = action.split()
        if len(parts) < 2:
            continue
        if parts[0] == 'd':
            if parts[1] == 'db' and len(parts) > 2:
                board.extend(split_cards(parts[2]))
                street_commit = [0.0] * n
            continue
        m = PL_RE.fullmatch(parts[0])
        if not m:
            continue
        i = int(m.group(1)) - 1
        verb = parts[1]
        if verb == 'sm':
            continue
        if verb == 'f':
            folded[i] = True
            continue
        if verb == 'cc':
            pay = max(0.0, max(street_commit) - street_commit[i])
            pay = min(pay, max(0.0, stacks0[i] - committed[i]))
            committed[i] += pay
            street_commit[i] += pay
        elif verb == 'cbr' and len(parts) > 2:
            amt = float(parts[2])
            pay = amt - street_commit[i]
            if pay < -1e-5:
                return None  # don't silently fix malformed PHH actions
            pay = min(pay, max(0.0, stacks0[i] - committed[i]))
            committed[i] += pay
            street_commit[i] += pay
        else:
            continue

        live = [p for p in range(n) if not folded[p]]
        allin = [p for p in live if stacks0[p] - committed[p] <= 1e-5]
        if len(live) == 2 and allin and len(board) < 5:
            # A call finalizes an all-in, as does a raise when both players
            # have become all-in. Do not count an uncalled shove.
            if verb == 'cc' or len(allin) == 2:
                return {
                    'players_idx': live,
                    'board_at_lock': board.copy(),
                    'street': {0:'preflop',3:'flop',4:'turn'}.get(len(board),'unknown'),
                    'allin_players_idx': allin,
                    'commit_bb_unscaled': min(committed[p] for p in live),
                }
    return None


def showdown_equity(hero: list[str], villain: list[str], board: list[str],
                    evaluator, trials=PREFLOP_TRIALS, seed=0) -> tuple[float, bool]:
    """Hero showdown equity (wins + half ties). Exact turn/flop, MC preflop."""
    from treys import Card
    exposed = hero + villain + board
    if len(set(exposed)) != len(exposed):
        raise ValueError('Duplicate known cards in a hand')
    need = 5 - len(board)
    if need <= 0:
        raise ValueError('Equity requires a board before the river')
    available = [c for c in DECK if c not in exposed]
    if len(board) in (3,4):
        runouts = itertools.combinations(available, need)
        is_exact = True
    else:
        rng = random.Random(seed)
        runouts = (rng.sample(available, need) for _ in range(trials))
        is_exact = False
    hero_c = [Card.new(c) for c in hero]
    villain_c = [Card.new(c) for c in villain]
    board_c = [Card.new(c) for c in board]
    score = 0.0
    total = 0
    for runout in runouts:
        community = board_c + [Card.new(c) for c in runout]
        h = evaluator.evaluate(hero_c, community)
        v = evaluator.evaluate(villain_c, community)
        score += 1.0 if h < v else 0.5 if h == v else 0.0
        total += 1
    return score / total, is_exact


def analyze_hand(hand: dict, source_file: str, evaluator) -> tuple[list[dict],dict]:
    """Return possible losing all-in examples and coverage counters."""
    from treys import Card
    from treys import Evaluator
    counts={'hands':1,'two_revealed':0,'heads_up_early_allin':0,
            'fully_observed_candidate':0,'skipped_invalid_cards':0}
    revealed = get_showdown_players(hand)
    if len(revealed) >= 2:
        counts['two_revealed'] = 1
    lock = find_heads_up_allin_lock(hand)
    if lock is None:
        return [],counts
    counts['heads_up_early_allin'] = 1
    pair = lock['players_idx']
    if any(p not in revealed for p in pair):
        return [],counts
    board=[]
    for a in hand.get('actions', []):
        parts=a.split()
        if len(parts)>=3 and parts[:2]==['d','db']:
            board.extend(split_cards(parts[2]))
    if len(board)!=5:
        return [],counts
    c0, c1 = revealed[pair[0]], revealed[pair[1]]
    if any(not re.fullmatch('[2-9TJQKA][cdhs]',c) for c in c0+c1+board):
        counts['skipped_invalid_cards']=1
        return [],counts
    if len(set(c0+c1+board))!=9:
        counts['skipped_invalid_cards']=1
        return [],counts
    counts['fully_observed_candidate'] = 1
    board_c=[Card.new(c) for c in board]
    s0=evaluator.evaluate([Card.new(c) for c in c0],board_c)
    s1=evaluator.evaluate([Card.new(c) for c in c1],board_c)
    if s0 == s1:
        return [],counts # tie; no loser
    loser=pair[0] if s0>s1 else pair[1] # lower treys score is better
    winner=pair[1] if loser==pair[0] else pair[0]
    players=hand.get('players',[])
    if max(pair)>=len(players):
        return [],counts
    bb=max(float(x or 0) for x in hand.get('blinds_or_straddles',[]))
    if abs(bb-1.0)>1e-8:
        return [],counts # specifically 100NL
    lock_board=lock['board_at_lock']
    try:
        equity, exact=showdown_equity(revealed[loser],revealed[winner],
                                      lock_board,evaluator,
                                      seed=int(hand.get('hand') or 0))
    except (ValueError, KeyError):
        counts['skipped_invalid_cards']=1
        return [],counts
    return [{
        'player':players[loser], 'opponent':players[winner],
        'hand_id':hand.get('hand'), 'source_file':source_file,
        'bb':bb, 'street_at_lock':lock['street'],
        'board_at_lock':''.join(lock_board),
        'final_board':''.join(board),
        'loser_cards':''.join(revealed[loser]),
        'winner_cards':''.join(revealed[winner]),
        'equity_at_lock':equity, 'equity_exact':exact,
        'min_contested_commit_bb':lock['commit_bb_unscaled']/bb,
    }],counts


def extract_events(data_dir: str | Path, verbose=True):
    import pandas as pd
    from treys import Evaluator
    evaluator=Evaluator()
    files=sorted(Path(data_dir).rglob('*.phhs'))+sorted(Path(data_dir).rglob('*.phh'))
    if not files:
        raise FileNotFoundError(f'No PHH hand histories in {data_dir}')
    events=[]
    tallies={'files':len(files),'hands':0,'two_revealed':0,
             'heads_up_early_allin':0,'fully_observed_candidate':0,
             'skipped_invalid_cards':0,'errors':0}
    for fi,path in enumerate(files,1):
        try:
            with open(path,'rb') as f:
                data=tomllib.load(f)
        except Exception as exc:
            tallies['errors']+=1
            if verbose: print(f'Cannot read {path.name}: {exc}')
            continue
        hands=[data] if 'actions' in data else data.values()
        for h in hands:
            if not isinstance(h,dict):continue
            if h.get('venue')!='Absolute Poker':continue
            bb=max([float(b or 0) for b in h.get('blinds_or_straddles',[])],default=0)
            if abs(bb-1)>1e-9:continue
            try:
                rows,count=analyze_hand(h,path.name,evaluator)
            except Exception as exc:
                tallies['errors']+=1
                if verbose and tallies['errors']<=10:print(f'Hand {h.get("hand")} error: {exc}')
                continue
            events.extend(rows)
            for k,v in count.items():tallies[k]+=v
        if verbose and (fi%25==0 or fi==len(files)):
            print(f'Processed {fi}/{len(files)} files; '
                  f'{tallies["fully_observed_candidate"]} evaluated all-in showdowns')
    return pd.DataFrame(events),tallies


def prepare_event_comparison(extracted, decisions, min_loss_bb=MIN_LOSS_BB,
                             equity_cutoff=BAD_BEAT_EQUITY):
    import pandas as pd
    from numpy import isclose
    # These columns were created by the ORIGINAL preprocessing notebook.
    required={'player','hand_id','hand_net_bb','session_key','hand_in_session'}
    missing=required-set(decisions.columns)
    if missing:raise ValueError(f'decisions.parquet missing: {sorted(missing)}')
    if extracted.empty:
        return pd.DataFrame()
    history=(decisions[['player','hand_id','hand_net_bb','session_key','hand_in_session']]
             .drop_duplicates(['player','hand_id']))
    events=extracted.merge(history,on=['player','hand_id'],how='left',validate='one_to_one',indicator=True)
    print('Matching raw all-in losses to existing decisions:', events['_merge'].value_counts().to_dict())
    events=events.drop(columns='_merge')
    events=events[events['hand_net_bb'].notna() & events['session_key'].notna()].copy()
    events['qualifying_big_loss']=events['hand_net_bb']<=-min_loss_bb
    events['bad_beat']=(events['equity_at_lock']>=equity_cutoff) & events['qualifying_big_loss']
    events['ordinary_allin_loss']=(events['equity_at_lock']<equity_cutoff) & events['qualifying_big_loss']
    events['event_type']='not_a_large_loss'
    events.loc[events['ordinary_allin_loss'],'event_type']='ordinary_allin_loss'
    events.loc[events['bad_beat'],'event_type']='bad_beat'
    return events


def compare_post_event_bets(events, decisions, scored_bets, window=WINDOW_HANDS):
    """Compare the original personal-score s>2 flag rate in next 1..window hands.

    Each bet is attributed to its MOST RECENT qualifying event (if any).
    If there are no bets after either type, returns an empty comparison.
    """
    import pandas as pd
    if events.empty:return pd.DataFrame(), pd.DataFrame()
    events=events[events.event_type.isin(['bad_beat','ordinary_allin_loss'])].copy()
    if events.empty:return pd.DataFrame(), pd.DataFrame()
    hand_lookup=(decisions[['player','hand_id','session_key','hand_in_session']]
                 .drop_duplicates(['player','hand_id']))
    bets=scored_bets.merge(hand_lookup,on=['player','hand_id'],how='left',validate='many_to_one',suffixes=('','_hands'))
    # If the score file already includes these columns, confirm and use those values.
    if 'session_key_hands' in bets.columns:
        assert (bets['session_key'].astype(str)==bets['session_key_hands'].astype(str)).all()
        bets=bets.drop(columns=['session_key_hands'])
    if 'hand_in_session_hands' in bets.columns:
        bets['hand_in_session']=bets['hand_in_session'].fillna(bets['hand_in_session_hands'])
        bets=bets.drop(columns=['hand_in_session_hands'])
    bets=bets.dropna(subset=['hand_in_session','session_key','player'])
    bets['hand_in_session']=bets['hand_in_session'].astype(int)
    events['hand_in_session']=events['hand_in_session'].astype(int)
    scores_cols=['player','session_key','hand_in_session','hand_id','personal_flag']
    if 'is_all_in' in bets.columns:scores_cols.append('is_all_in')
    ev=events[['player','session_key','hand_in_session','hand_id','event_type',
               'equity_at_lock','hand_net_bb']].rename(columns={
                   'hand_in_session':'event_hand_in_session',
                   'hand_id':'event_hand_id'})
    # Must sort ON numeric asof key before merge_asof, even when using 'by'.
    ranked_bets=bets[scores_cols].sort_values('hand_in_session',kind='stable').copy()
    ev=ev.sort_values('event_hand_in_session',kind='stable')
    joined=pd.merge_asof(ranked_bets,ev,left_on='hand_in_session',
                         right_on='event_hand_in_session',
                         by=['player','session_key'],direction='backward',
                         allow_exact_matches=False)
    joined['hands_since_event']=joined['hand_in_session']-joined['event_hand_in_session']
    after=joined[joined.hands_since_event.between(1,window)].copy()
    if after.empty:return pd.DataFrame(),after
    result=after.groupby('event_type',as_index=False).agg(
        betting_decisions=('personal_flag','size'),
        flagged_bets=('personal_flag','sum'),
        flag_rate=('personal_flag','mean'),
        unique_player_sessions=('session_key','nunique'),
        unique_event_hands=('event_hand_id','nunique'),
    )
    result['flag_rate_percent']=(100*result.flag_rate).round(3)
    return result,after


def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--handhq',required=True)
    ap.add_argument('--decisions',default='decisions.parquet')
    ap.add_argument('--scores',default='personal_bet_scores.csv.gz',
                    help='From ORIGINAL poker_personal_model.py; has personal_flag for s>2')
    ap.add_argument('--output',default='badbeat_100nl_outputs')
    ap.add_argument('--min-loss-bb',type=float,default=MIN_LOSS_BB)
    ap.add_argument('--equity-cutoff',type=float,default=BAD_BEAT_EQUITY)
    ap.add_argument('--window',type=int,default=WINDOW_HANDS)
    args=ap.parse_args()
    import pandas as pd
    out=Path(args.output);out.mkdir(parents=True,exist_ok=True)
    extracted,coverage=extract_events(args.handhq)
    pd.DataFrame([coverage]).to_csv(out/'coverage.csv',index=False)
    extracted.to_csv(out/'revealed_heads_up_allins.csv',index=False)
    print('Coverage:',json.dumps(coverage,indent=2))
    if extracted.empty:
        print('No fully observed heads-up all-in losses found. Cannot compute comparison.')
        return
    decisions=pd.read_parquet(args.decisions)
    events=prepare_event_comparison(extracted,decisions,args.min_loss_bb,args.equity_cutoff)
    events.to_csv(out/'classified_losses.csv',index=False)
    print('Event classes:',events.event_type.value_counts().to_dict())
    if not Path(args.scores).exists():
        print(f'No score file at {args.scores}; events saved, but no post-event anomaly comparison yet.')
        return
    scores=pd.read_csv(args.scores)
    result,after=compare_post_event_bets(events,decisions,scores,args.window)
    result.to_csv(out/'post_event_comparison.csv',index=False)
    after.to_csv(out/'post_event_bet_rows.csv',index=False)
    print('Post-event comparison:',result.to_string(index=False) if not result.empty else 'No eligible events')
    print('Outputs:',out.resolve())


if __name__=='__main__':
    main()


### Run the equity classification and post-event comparison

For reproducibility, default cutoff is `0.70` (at least 70% equity). Change `BB_EQUITY_THRESHOLD` to `0.80` 

In [ ]:
# Run after the wins section has exported its original 100NL s>2 scores.
# Use 0.70 to reproduce the earlier analysis definition; use 0.80 for the
# stricter >80% threshold described in a later presentation.
BB_EQUITY_THRESHOLD = 0.70
BB_MIN_LOSS = 30
BB_NEXT_HANDS = 5

_BB_CMD = [
    _bb_sys.executable, 'badbeat_100NL_analysis.py',
    '--handhq', str(BB_HAND_HISTORY_FOLDER),
    '--decisions', str(WIN_DECISIONS_100NL),
    '--scores', WIN_ADDITION_OUTPUTS['badbeat_scores'],
    '--output', 'badbeat_100NL_outputs',
    '--equity-cutoff', str(BB_EQUITY_THRESHOLD),
    '--min-loss-bb', str(BB_MIN_LOSS),
    '--window', str(BB_NEXT_HANDS),
]
_bb_subprocess.run(_BB_CMD, check=True)


In [ ]:
# Side-by-side display of the actual bad-beat rates, if there are enough
# observed events. No percentages are hard-coded.
import pandas as _bb_pd
from IPython.display import display as _bb_display
_BB_FILE = _BBPath('badbeat_100NL_outputs/post_event_comparison.csv')
if _BB_FILE.exists() and _BB_FILE.stat().st_size:
    _BB_TABLE = _bb_pd.read_csv(_BB_FILE)
    _bb_display(_BB_TABLE)
    if not _BB_TABLE.empty:
        import matplotlib.pyplot as _bb_plt
        _fig, _ax = _bb_plt.subplots(figsize=(8,4))
        _ax.bar(_BB_TABLE['event_type'],100*_BB_TABLE['flag_rate'])
        _ax.set_ylabel('Flagged bet rate (%, s > 2)')
        _ax.set_title('100NL: unusual bets after large all-in losses')
        _fig.tight_layout()
        _bb_plt.show()
else:
    print('No post-event comparison available: check data coverage and saved results.')


## Output files and GitHub notes

After running, outputs are stored under:
- `wins_100NL_outputs/` — descriptive comparisons, fitted-model results, and 100NL personal scores.
- `badbeat_100NL_outputs/` — coverage, identified all-in events, classified loss types, and post-event anomaly rates.